# Topic modeling with BERTopic

In [9]:
import sqlite3
import pandas as pd
import numpy as np
from tqdm import tqdm

## Prepare data

In [10]:
db_path = "../data/processed/dblp.db"

conn = sqlite3.connect(db_path)

query = """
SELECT id, title, year, venue, type
FROM papers
WHERE title IS NOT NULL
"""

df = pd.read_sql_query(query, conn)

conn.close()

df.head()

,id,title,year,venue,type
0,series/sapere/Freed13,Practical Introspection as Inspiration for AI.,2011,PT-AI,inproceedings
1,series/sapere/Steiner13,C.S. Peirce and Artificial Intelligence: Histo...,2011,PT-AI,inproceedings
2,series/sapere/Sandberg13,Feasibility of Whole Brain Emulation.,2011,PT-AI,inproceedings
3,series/sapere/BerkeleyR13,Machine Mentality?,2011,PT-AI,inproceedings
4,series/sapere/NasutoB13,Of (Zombie) Mice and Animats.,2011,PT-AI,inproceedings


In [11]:
df_sample = df.sample(n=200000, random_state=42)
docs = df_sample["title"].tolist()

## BERTopic

In [12]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

### Parameteres

In [13]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist = 0.0,
    metric="cosine",
    low_memory=True,
    random_state=42,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=300,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

custom_stop = list(ENGLISH_STOP_WORDS) + [
    "based", "using", "novel", "efficient", "approach", "method",
    "paper", "proposed", "new", "via", "toward", "towards", "use",
    "used", "study", "system", "systems",
]

vectorizer_model = CountVectorizer(
    stop_words=custom_stop,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9
)

representation_model = KeyBERTInspired()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2345.27it/s]


### Train

In [14]:
embeddings = embedding_model.encode(
    docs,
    batch_size=512,
    normalize_embeddings=True,
    show_progress_bar=True,
)
np.save("../results/embeddings.npy", embeddings)


Batches: 100%|██████████| 391/391 [12:23<00:00,  1.90s/it]


In [15]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    representation_model=representation_model,
    top_n_words=10,
    nr_topics=50,
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

2026-04-26 01:35:18,310 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-26 01:39:38,490 - BERTopic - Dimensionality - Completed ✓
2026-04-26 01:39:38,494 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-26 01:39:51,484 - BERTopic - Cluster - Completed ✓
2026-04-26 01:39:51,484 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-26 01:39:55,812 - BERTopic - Representation - Completed ✓
2026-04-26 01:39:55,815 - BERTopic - Topic reduction - Reducing number of topics
2026-04-26 01:39:55,953 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-26 01:40:05,630 - BERTopic - Representation - Completed ✓
2026-04-26 01:40:05,646 - BERTopic - Topic reduction - Reduced number of topics from 128 to 50


In [16]:
new_topics = topic_model.reduce_outliers(
    docs,
    topics,                     # oryginalne topics z fit_transform
    strategy="embeddings",
    embeddings=embeddings,      # twoje 200k embeddingi numpy
    threshold=0.2,              # cosine similarity – obniż do 0.3 jeśli nadal dużo outlierów
)

topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer_model)
topic_model.save('../results/bertopic_model')

2026-04-26 01:40:07,121 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-04-26 01:40:11,926 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


### Results

In [17]:
# Topic list
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,720,-1_thermal_cu_cell_magnetic,"[thermal, cu, cell, magnetic, cyberbullying, t...",[Interference Management in Visible Light Comm...
1,0,14285,0_object_segmentation_radar_object detection,"[object, segmentation, radar, object detection...",[Coding Convolutional Neural Networks as Spect...
2,1,12455,1_wireless_mimo_radio_sensor networks,"[wireless, mimo, radio, sensor networks, wirel...",[Energy efficiency resource allocation in down...
3,2,7410,2_education_students_reality_teaching,"[education, students, reality, teaching, stude...","[Live Captions in Virtual Reality (VR)., Spati..."
4,3,7986,3_adversarial_neural networks_visual_generative,"[adversarial, neural networks, visual, generat...",[NaturalBench: Evaluating Vision-Language Mode...
5,4,7614,4_language models_large language_text_sentiment,"[language models, large language, text, sentim...",[JTSG: A joint term-sentiment generator for as...
6,5,6382,5_privacy_secure_encryption_security,"[privacy, secure, encryption, security, authen...",[An improved secure certificateless public-key...
7,6,6146,6_segmentation_images_ct_medical,"[segmentation, images, ct, medical, mri, imagi...",[Diffusion-Based Data Augmentation for Medical...
8,7,7221,7_scheduling_gpu_fpga_storage,"[scheduling, gpu, fpga, storage, cache, hardwa...",[A GPU-Based Parallel Reduction Implementation...
9,8,4852,8_dc_electric_voltage_converter,"[dc, electric, voltage, converter, wind, batte...",[Power Smoothing in MCT System with Supercapac...


In [18]:
# Words for specific topic
topic_model.get_topic(5)

[('privacy', np.float64(0.08509300938266251)),
 ('secure', np.float64(0.05810975809507396)),
 ('encryption', np.float64(0.05714904166513591)),
 ('security', np.float64(0.051143712955325805)),
 ('authentication', np.float64(0.04071186317281107)),
 ('key', np.float64(0.03651032491579972)),
 ('privacy preserving', np.float64(0.03343269995398265)),
 ('preserving', np.float64(0.03290673949881768)),
 ('attacks', np.float64(0.028178232789656563)),
 ('private', np.float64(0.026899433287596405))]

In [19]:
# Topic visualization
topic_model.visualize_topics()

In [20]:
# Topic relations
topic_model.visualize_hierarchy()

In [21]:
# Most important words
topic_model.visualize_barchart(top_n_topics=10)

In [22]:
df_sample["topic"] = topics

In [23]:
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps=df_sample["year"].tolist()
)

topic_model.visualize_topics_over_time(topics_over_time)

16it [00:08,  1.99it/s]


## Topic labeling

In [24]:
import os
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ── 1. Ładowanie modelu BERTopic ──────────────────────────────
BERTOPIC_PATH = "../results/bertopic_model"
EMBEDDINGS_PATH = "../results/embeddings.npy"

print("Ładowanie modelu BERTopic...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic.load(BERTOPIC_PATH, embedding_model=embedding_model)
print("Model BERTopic wczytany.")

# ── 2. Embeddingi ─────────────────────────────────────────────
if os.path.exists(EMBEDDINGS_PATH):
    print("Wczytywanie embeddingów z pliku...")
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f"Embeddingi wczytane: {embeddings.shape}")
else:
    print("Generowanie embeddingów (może potrwać ~10 min)...")
    embeddings = embedding_model.encode(
        df_sample["title"].tolist(),
        batch_size=512,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    np.save(EMBEDDINGS_PATH, embeddings)
    print("Embeddingi zapisane.")

# ── 3. Przypisanie tematów ────────────────────────────────────
print("Przypisywanie tematów...")
topics, _ = topic_model.transform(
    df_sample["title"].tolist(),
    embeddings=embeddings,
)

new_topics = topic_model.reduce_outliers(
    df_sample["title"].tolist(),
    topics,
    strategy="embeddings",
    embeddings=embeddings,
    threshold=0.3,
)
print(f"Tematów przypisanych: {len(set(new_topics)) - (1 if -1 in new_topics else 0)}")
print(f"Outlierów: {new_topics.count(-1) if isinstance(new_topics, list) else (new_topics == -1).sum()}")

# ── 4. Ładowanie TinyLlama ────────────────────────────────────
TINYLLAMA_PATH = "../results/tinyllama_pipeline"

print("Ładowanie modelu do etykiet...")
if os.path.exists(TINYLLAMA_PATH):
    generator = pipeline("text-generation", model=TINYLLAMA_PATH, device=-1, do_sample=False)
else:
    generator = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", device=-1, do_sample=False)
    generator.save_pretrained(TINYLLAMA_PATH)
print("TinyLlama wczytany.")

# ── 5. Generowanie etykiet ────────────────────────────────────
def generate_label(topic_id):
    if topic_id == -1:
        return "Outliers"
    
    keywords = [w for w, _ in topic_model.get_topic(topic_id)[:8]]
    keywords_str = ", ".join(keywords)
    
    prompt = f"""<|system|>
You are a computer science researcher. Return only a short topic label (2-4 words), nothing else.</s>
<|user|>
Keywords: {keywords_str}
Label:</s>
<|assistant|>"""
    
    output = generator(
        prompt,
        return_full_text=False,
        max_new_tokens=10,
        pad_token_id=generator.tokenizer.eos_token_id,
    )[0]["generated_text"]
    
    label = output.strip().split("\n")[0].strip().strip('"').strip("'")
    return label if label else "Unknown Topic"

info = topic_model.get_topic_info()
topic_labels = {}

print("Generowanie etykiet dla tematów...")
for _, row in info.iterrows():
    tid = row["Topic"]
    if tid == -1:
        topic_labels[-1] = "Outliers"
        continue
    label = generate_label(tid)
    topic_labels[tid] = label
    print(f"  Topic {tid:3d} ({row['Count']:5,} docs): {label}")

# ── 6. Przypisanie do DataFrame ───────────────────────────────
topic_model.set_topic_labels(topic_labels)

df_sample = df_sample.copy()  # unika SettingWithCopyWarning
df_sample["topic_id"] = new_topics
df_sample["topic_label"] = df_sample["topic_id"].map(topic_labels)

# ── 7. Podgląd ────────────────────────────────────────────────
print("\n── Rozkład tematów ──────────────────────────────────────")
print(df_sample[df_sample["topic_id"] != -1]["topic_label"].value_counts().head(20).to_string())

print("\n── Przykładowe dokumenty ────────────────────────────────")
print(df_sample[df_sample["topic_id"] != -1][["title", "topic_label"]].sample(10).to_string())

Ładowanie modelu BERTopic...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3840.84it/s]


Model BERTopic wczytany.
Wczytywanie embeddingów z pliku...
Embeddingi wczytane: (200000, 384)
Przypisywanie tematów...


2026-04-26 01:41:03,704 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-26 01:41:04,411 - BERTopic - Dimensionality - Completed ✓
2026-04-26 01:41:04,412 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-26 01:41:30,726 - BERTopic - Cluster - Completed ✓


Tematów przypisanych: 49
Outlierów: 5759
Ładowanie modelu do etykiet...


...

  Topic  45 (1,019 docs): Keywords: disaster, emergency,


[transformers] Both `max_new_tokens` (=10) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Topic  46 (1,139 docs): Keywords: causal, counterfactual


[transformers] Both `max_new_tokens` (=10) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Topic  47 (  808 docs): Keywords: antenna, band, wide
  Topic  48 (1,045 docs): Keywords: eye, gaze, eye

── Rozkład tematów ──────────────────────────────────────
topic_label
Keywords: object, segmentation, rad           14153
Keywords: wireless, mimo, radio               12393
Keywords: adversarial, neural networks,        7954
Keywords: language models, large language,     7548
Keywords: education, students, reality,        7216
Keywords: scheduling, gpu,                     7121
Keywords: matrix, gaussian, rank               6487
Keywords: privacy, secure, encryption          6314
Keywords: segmentation, images, c              6023
Keywords: business, innovation, social         5787
Keywords: graphs, social networks, graph       5208
Keywords: recommendation, ontology, recomm     5208
Keywords: code, requirements, checking,        4913
Keywords: dc, electric, voltage                4735
Keywords: nonlinear, stability, delay          4597
Keywords: health, covid, cov              